In [1]:
%load_ext autoreload
%autoreload 2

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from importlib import reload

import mh
import mcmc_plots
import baselines
import downstream
reload(mh)
reload(mcmc_plots)
reload(baselines)
reload(downstream)
from mh import generate_synthetic_data, MHSampler, SliceData
from mcmc_plots import MCMCPlotter
from baselines import run_all_baselines, add_mcmc_result, print_comparison_table
from downstream import PosteriorAnalysis

# Synthetic Data (Simulation)

In [3]:
print("Generating synthetic data …")
data, truth = generate_synthetic_data(N=2500, P=100, L=20, seed=42)
print(f"  Y shape: {data.Y.shape},  X_ref shape: {data.X_ref.shape}")

Generating synthetic data …
  Y shape: (2500, 100),  X_ref shape: (2500, 20)


## Bayesian Hierarchical model

In [4]:
print("\nCompiling numba kernels ...")
warm = MHSampler(data, n_iter=3, burn_in=0, seed=0)
warm.run(verbose=False)
print("  done.")

proposal_sd = {"beta": 0.1, "gamma": 5, "log_phi": 0.1}

sampler = MHSampler(
    data, n_iter=5000, burn_in=1000, thin=10, seed=99,
    proposal_sd=proposal_sd,
)
print(f"\nRunning sampler ({sampler.n_iter} iters) ...\n")
sampler.run(verbose=True)


Compiling numba kernels ...
  done.

Running sampler (5000 iters) ...

Iter    500/5000  lp/obs=-4.1229  g_bar=0.313  [34.1s elapsed]
Iter   1000/5000  lp/obs=-4.1077  g_bar=0.316  [65.0s elapsed]
Iter   1500/5000  lp/obs=-4.1080  g_bar=0.322  [98.2s elapsed]
Iter   2000/5000  lp/obs=-4.1079  g_bar=0.316  [131.5s elapsed]
Iter   2500/5000  lp/obs=-4.1080  g_bar=0.314  [165.8s elapsed]
Iter   3000/5000  lp/obs=-4.1081  g_bar=0.320  [199.9s elapsed]
Iter   3500/5000  lp/obs=-4.1078  g_bar=0.320  [233.7s elapsed]
Iter   4000/5000  lp/obs=-4.1078  g_bar=0.322  [268.2s elapsed]
Iter   4500/5000  lp/obs=-4.1078  g_bar=0.315  [303.0s elapsed]
Iter   5000/5000  lp/obs=-4.1077  g_bar=0.321  [337.9s elapsed]

Total: 337.91s  (67.6 ms/iter)

--- Acceptance rates ---
      beta: 0.716  (7523112/10500000)
     gamma: 0.040  (83778/2100000)
       phi: 0.365  (182485/500000)


In [5]:
plotter = MCMCPlotter(sampler, truth=truth)
plotter.save_all("plots/simulated/")

Saved plots to /oscar/data/yma16/schang59/bayesian/plots/simulated/


## Baselines

In [6]:
results = run_all_baselines(data, truth=truth)
results = add_mcmc_result(results, sampler, data, truth)

       KNN Spatial Avg:  ll/obs=-5.5898  mse=2.13±0.57  mae=1.02±0.12  (0.06s)
        Ridge log(y+1):  ll/obs=-7.8926  mse=2.10±0.50  mae=1.11±0.12  (0.14s)
        Lasso log(y+1):  ll/obs=-8.0381  mse=2.13±0.50  mae=1.12±0.12  (0.15s)


## Metrics Comparison

In [7]:
print_comparison_table(results)

Method                     LL/obs    logMSE/spot    logMAE/spot      F1     AUC
----------------------------------------------------------------------------------
KNN Spatial Avg           -5.5898      2.13±0.57      1.02±0.12       —       —
Ridge log(y+1)            -7.8926      2.10±0.50      1.11±0.12   0.520       —
Lasso log(y+1)            -8.0381      2.13±0.50      1.12±0.12   0.772   0.941
Bayesian Hierarch         -4.1045      1.18±0.29      0.76±0.07   0.968   0.986


# Real Data

In [8]:
Y = np.loadtxt('Y_mouse.csv', delimiter=',')
T = np.loadtxt('T_target_mouse.csv', delimiter=',')
Y_ref = np.loadtxt('X_ref_mouse.csv', delimiter=',')
T_ref = np.loadtxt('T_ref_mouse.csv', delimiter=',')

# Y = np.loadtxt('Y.csv', delimiter=',')
# T = np.loadtxt('T_target.csv', delimiter=',')
# Y_ref = np.loadtxt('X_ref.csv', delimiter=',')
# T_ref = np.loadtxt('T_ref.csv', delimiter=',')

In [9]:
data = SliceData(Y=Y, T=T)
data.set_reference(Y_ref, T_ref, L=20)

## Bayesian Hierarchical model

In [19]:
print("\nCompiling numba kernels ...")
warm = MHSampler(data, n_iter=3, burn_in=0, seed=0)
warm.run(verbose=False)
print("  done.")

proposal_sd = {"beta": 0.5, "gamma": 5, "log_phi": 0.5}

sampler = MHSampler(
    data, n_iter=25000, burn_in=10000, thin=10, seed=99,
    proposal_sd=proposal_sd,
)
print("\nRunning sampler ...\n")
sampler.run(verbose=True)


Compiling numba kernels ...
  done.

Running sampler ...

Iter   2500/25000  lp/obs=-0.5860  g_bar=0.260  [110.0s elapsed]
Iter   5000/25000  lp/obs=-0.5859  g_bar=0.263  [223.5s elapsed]
Iter   7500/25000  lp/obs=-0.5859  g_bar=0.253  [336.5s elapsed]
Iter  10000/25000  lp/obs=-0.5857  g_bar=0.260  [445.5s elapsed]
Iter  12500/25000  lp/obs=-0.5856  g_bar=0.264  [558.9s elapsed]
Iter  15000/25000  lp/obs=-0.5857  g_bar=0.265  [674.9s elapsed]
Iter  17500/25000  lp/obs=-0.5859  g_bar=0.253  [791.5s elapsed]
Iter  20000/25000  lp/obs=-0.5857  g_bar=0.269  [907.1s elapsed]
Iter  22500/25000  lp/obs=-0.5858  g_bar=0.259  [1023.1s elapsed]
Iter  25000/25000  lp/obs=-0.5857  g_bar=0.272  [1132.4s elapsed]

Total: 1132.42s  (45.3 ms/iter)

--- Acceptance rates ---
      beta: 0.591  (16244752/27500000)
     gamma: 0.031  (171292/5500000)
       phi: 0.301  (752348/2500000)


In [20]:
from mcmc_plots import MCMCPlotter

plotter = MCMCPlotter(sampler)
plotter.save_all("plots/mouse/")

Saved plots to /oscar/data/yma16/schang59/bayesian/plots/mouse/


In [21]:
import pickle
with open('mouse_estimates.pkl', 'wb') as f:
    pickle.dump(sampler, f)

In [22]:
import pickle
with open('mouse_estimates.pkl', 'rb') as f:
    sampler = pickle.load(f)

## Baselines

In [23]:
results = run_all_baselines(data)
results = add_mcmc_result(results, sampler, data)

       KNN Spatial Avg:  ll/obs=-0.5210  mse=0.16±0.09  mae=0.21±0.06  (0.07s)
        Ridge log(y+1):  ll/obs=-0.7122  mse=0.19±0.10  mae=0.23±0.06  (0.11s)
        Lasso log(y+1):  ll/obs=-0.8780  mse=0.19±0.11  mae=0.20±0.06  (0.14s)


## Metrics comparison

In [24]:
print_comparison_table(results)

Method                     LL/obs    logMSE/spot    logMAE/spot      F1     AUC
----------------------------------------------------------------------------------
KNN Spatial Avg           -0.5210      0.16±0.09      0.21±0.06       —       —
Ridge log(y+1)            -0.7122      0.19±0.10      0.23±0.06       —       —
Lasso log(y+1)            -0.8780      0.19±0.11      0.20±0.06       —       —
Bayesian Hierarch         -0.5847      0.23±0.16      0.30±0.11       —       —


## Downstream analysis

In [18]:
gene_names = np.loadtxt('gene_names_mouse.csv', delimiter=',', dtype=str) 

In [31]:
pa = PosteriorAnalysis(sampler, data, gene_names=gene_names)
pa.save_all("downstream/mouse/")

Downstream plots finished
